**Étape 1 : Préparer l'Environnement**

In [7]:
import os

os.environ["LANGCHAIN_TRACING_V2"] = "True"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.langchain.com"


In [5]:
os.environ["USER_AGENT"] = "MyLangChainApp/1.0"

In [2]:
pip install -r /requirements.txt

In [18]:
!pip install langgraph

**Étape 2 : Créer la Base de Connaissances (RAG)**


In [3]:
# Charger les données

from langchain.document_loaders import WebBaseLoader

loader = WebBaseLoader(["https://www.louisbouchard.ca/apprendre-ia", "https://www.plateya.fr/blog/detail/formation-intelligence-artificielle-pour-debutants-guide-2025"])
documents = loader.load()

In [8]:
# Découper les documents

from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.split_documents(documents)

In [9]:
# Vectoriser les documents

from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma

embeddings = HuggingFaceEmbeddings()
vectorstore = Chroma.from_documents(texts, embeddings)

/tmp/ipython-input-3549223593.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings()
/tmp/ipython-input-3549223593.py:6: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://hugg

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
# Créer le retriever

from langchain.chains import RetrievalQA

retriever = vectorstore.as_retriever()

**Étape 3 : Définir l'État de l'Agent**


In [11]:
class AgentState:
    def __init__(self):
        self.history = []

    def add_message(self, role, content):
        self.history.append({"role": role, "content": content})

**Étape 4 : Construire les Nœuds du Graphe**


In [12]:
# Nœud Initial (ai_assistant)

def ai_assistant(state, question):
    # Logique pour décider si un outil est nécessaire
    if needs_tool(question):
        return "retrieve"
    else:
        return "generate"

In [13]:
# Nœud de Récupération (retrieve)

def retrieve(state, question):
    docs = retriever.get_relevant_documents(question)
    return docs

In [14]:
# Fonction de Décision (grade_documents)

def grade_documents(docs):
    # Utilise un LLM pour obtenir un score de pertinence
    relevance_score = llm.evaluate(docs)
    return relevance_score > 0.5

In [15]:
# Nœud de Génération (generate)

def generate(state, docs):
    answer = llm.generate(docs)
    state.add_message("assistant", answer)
    return answer

In [16]:
# Nœud de Réécriture (rewrite)

def rewrite(state, question):
    new_question = llm.rewrite(question)
    state.add_message("user", new_question)
    return new_question

**Étape 5 : Assembler le Graphe**


In [22]:
from langgraph.graph import StateGraph

# Define your state schema here
# For example, using the AgentState class you defined earlier
# from typing import TypedDict, List
#
# class AgentState(TypedDict):
#     history: List[dict]
#     documents: List[str]


graph = StateGraph(state_schema=AgentState)

In [23]:
# Ajouter les nœuds

graph.add_node("ai_assistant", ai_assistant)
graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)
graph.add_node("rewrite", rewrite)

In [25]:
# Connecter les nœuds

graph.add_conditional_edges(
    "ai_assistant",  # Start node
    lambda x: "retrieve" if needs_tool(x['question']) else "generate", # Conditional function
    {
        "retrieve": "retrieve",
        "generate": "generate",
    }
)

graph.add_conditional_edges(
    "retrieve",  # Start node
    lambda x: "generate" if grade_documents(x['documents']) else "rewrite", # Conditional function
    {
        "generate": "generate",
        "rewrite": "rewrite",
    }
)

graph.add_edge("rewrite", "ai_assistant") # Unconditional transition

**Étape 6 : Compiler et Tester l'Application**


In [27]:
from langgraph.graph import END, START

# Compile le graphe

graph.add_edge(START, "ai_assistant") # Add an edge from START to the first node

app = graph.compile()

In [30]:
# Tester l'application

# state = AgentState() # Removed as invoke expects a dictionary-like object
question = "Quelles sont les avancées majeures de l'IA ?"
response = app.invoke({"question": question}) # Pass initial state as a dictionary
print(response)

TypeError: ai_assistant() missing 1 required positional argument: 'question'

**Action Required:** Define the `needs_tool` function.

You need to add a code cell that defines the `needs_tool` function. This function should take the user's question as input and return `True` if a tool (like the retriever) is needed to answer the question, and `False` otherwise. For example: